# Introduction

Football (in USA: soccer) is by far the most popular sport in the world. Billions of people have watched several games, and many people are betting on the outcome of a football game.

Classically, there are three possible outcomes of a football game: the home team wins (H), a draw (D), or the away team wins (A). This sounds like an interesting classification task for machine learning models, right? :-) 

The worldwide sport betting market is growing constantly. Only in Germany in 2016, there were more than 5 Billion € of bets spent (https://de.statista.com/infografik/10125/daten-zum-thema-sportwetten/).

What if you can predict a match outcome by using machine learning models with a good accuracy? You would be the richest person on the planet, I guess :-) Of course, if it would be possible with a very high accuracy, the game would be boring and betting companies would not operate because of the lack of a business model.

But still, as I'm really interested in football, I want to try to build a machine learning model that predicts a match outcome with the highest possible accuracy. So let's get started!

# Conclusion Summary

Before you want to read the long notebook, here's a short summary of what I did:

Problem Definition: The goal is to predict match results. Possible results are home team win (h), draw (d), and away team win (a). So we are dealing with a multi-class prediction problem. Accuracy score represents the evaluation metric in this case. 

The dataset contains information that can be clustered by following clusters:
- Main Data
- Match Statistics
- 1X2 Betting Odds
- Over/Under Betting Odds
- Asian Handicap Betting Odds

Additionally, the following features were created:
- short-term match statistics (3 games)
- middle-term match statistics  (10 games)
- long-term match statistics long-term (30 games)

The dataset was preprocessed by following steps:
- Handling Missing Values
- Handling Outliers
- Dummyfication of categorical variables
- Handling Imbalanced Data
- Partitioning into Train and Test Sets
- Scaling Data

The following possibilities were tested to achieve the highest possible accuracy:
- different algorithms (logistic regression, random forest, decision tree, k-nearest neighbors)
- hyperparameter tuning
- oversampling with two different methods (smote and simple random oversampling)
- different splitting strategies regarding train_test_split (random split and splitting by seasons)
- considering additional new features that contain information about the form of the team for multiple in-game statistics
- considering various betting odds as features (usual HDA-odds, over/under-odds, asian handicap odds)
- various feature selection methods (univariate correlation (KBest), PCA (selecting best PC's))

The best model with an out-of-sample accuracy of slightly 0.53 is a random forest model where the input data includes odd-features and features that describe the short term form of a team selected by feature selection method KBest. The model parameters are {'max_depth': 5, 'n_estimators': 50}.

Compared to the benchmark model (predict always home team win (class "h") with accuracy of 0.46), the more sophisticated model predicts better by 7 percentage points.

# Libraries

Loading all python libraries that I will need for the analysis.

In [ ]:
# I will use pandas dataframes as the main data storage objects
import pandas as pd
import numpy as np

# for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# for reading and getting the data
import sqlite3
import urllib

# Various Machine Learning Model objects
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# For preparing data ready to be used in ML-models abd evaluation of ML-models
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline, make_pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA

# Functions

Storing all self-created functions that I will use for the analysis grouped by the purpose of use.

## Load Data

In [ ]:
def tables_in_sqlite_db(conn):
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [
        v[0] for v in cursor.fetchall()
        if v[0] != "sqlite_sequence"
    ]
    cursor.close()
    return tables

# Source: https://techoverflow.net/2019/10/14/how-to-list-tables-in-sqlite3-database-in-python/

## Data Examination

In [ ]:
def plot_share_of_data(df, column_to_groupby):
    data = fd.value_counts(column_to_groupby, sort=False, normalize=True).reset_index().rename({0:"share"}, axis=1)

    fig = plt.figure(figsize=(15,7))

    ax = plt.barh(data[column_to_groupby], data["share"])

    plt.ylabel(column_to_groupby)
    plt.xlabel("Share of Data")
    plt.show()

## Data Preparation

In [ ]:
def display_na_cols(df):
    with pd.option_context("display.min_rows", 50, "display.max_rows", 200, "display.max_columns", 5):
        display(df[df.columns[df.isna().any()]].isnull().sum())

In [ ]:
def group_col_and_count_na(df, column_to_groupby):

    """
    df: pandas dataframe
    column_to_groupby: column name of column in df as string
    """

    output = df.drop(column_to_groupby, 1).isna().groupby(df[column_to_groupby], sort=False).sum().reset_index()   

    return output

In [ ]:
def custom_window_function(df, sort_col, sort_asc, group_col, target_col, target_agg, lag, new_col_name):
    
    output = df\
                .sort_values(by=sort_col, ascending=sort_asc)\
                .set_index("index_id")\
                .groupby(group_col)\
                [[target_col]]\
                .rolling(lag)\
                .agg(target_agg)\
                .groupby(group_col).shift(1)\
                .reset_index()\
                .rename({target_col:new_col_name},axis=1)\
                .drop(group_col, axis=1)\
                .merge(df, on="index_id", how="inner")
    
    return output

## Modelling

In [ ]:
# Storing all necessary stratified grid search steps in one function so that it will be easier to access these steps later on.
def cv_procedure(estimator,param_grid,X,y, scoring="accuracy"):

    ''' 
    Input:
    estimator = sklearn classifier object
    param_grid = dictionary with all hyperparameters that should be tested in grid search
    X, y = Input and target features

    Output: pandas dataframe
    '''

    # Splitting the data into 5 Folds, where all experiments are repeated 2 times. (higher numbers would cause really high amount of calculation time so I decided to stick to this combination.)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

    # Performing Grid Search based on the folds that were defined above
    search = GridSearchCV(estimator=estimator, param_grid=param_grid, scoring=scoring, cv=cv)
    search.fit(X, y)
    ## The algorithms will be evaluated by the evaluation metric that is defined in the scroing variable.

    # Creating a pandas dataframe with all search results and transforming it.
    results_df = pd.DataFrame(search.cv_results_).sort_values(by=["rank_test_score"])
    results_df = results_df.set_index(
                            results_df["params"].apply(lambda x: "_".join(str(val) for val in x.values()))
                            )\
                                .rename_axis("kernel")
    
    return results_df[["params", "rank_test_score", "mean_test_score", "std_test_score"]]

# This experimental setting was inspired and partly copied from:
# https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html#sphx-glr-auto-examples-model-selection-plot-grid-search-stats-py

In [ ]:
# Storing all necessary stratified grid search steps in one function so that it will be easier to access these steps later on.
def cv_procedure_sample(estimator,param_grid,X,y,sampler, alg_name, name_add, scoring="accuracy"):
    
    # Splitting the data into 5 Folds, where all experiments are repeated 2 times. (higher numbers would cause really high amount of calculation time so I decided to stick to this combination.)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
    
    # Creating a Pipeline
    pipeline = make_pipeline(sampler, estimator)
    new_params = {name_add + key: param_grid[key] for key in param_grid}

    # Performing Grid Search based on the folds that were defined above
    search = GridSearchCV(pipeline, param_grid=new_params, scoring=scoring, cv=cv)
    search.fit(X, y)
    ## The algorithms will be evaluated by the evaluation metric that is defined in the scroing variable.

    y_test_predict = search.predict(X_test)
    final_list = [alg_name, search.best_params_, accuracy_score(y_test, y_test_predict), balanced_accuracy_score(y_test, y_test_predict), confusion_matrix(y_test_predict, y_test)]
    
    return final_list

## Evaluation

In [ ]:
# Create an evaluation class to have a python object where we can store the evaluation information that we need for every algorithm with its best hyperparameters
class evaluation:
   def __init__(self, cv_results, sklearn_classifier, estimator, X_train, y_train, X_test, y_test):
         self.cv_results = cv_results
         self.classifier = sklearn_classifier
         self.estimator = estimator
         self.X_train = X_train
         self.y_train = y_train
         self.X_test = X_test
         self.y_test = y_test
   
   def evaluate(self):
        self.best_param = self.cv_results.reset_index().iloc[0,1]
        self.clf = self.classifier(**dict(self.estimator.get_params(), **self.best_param))
        self.model = self.clf.fit(X=self.X_train, y=self.y_train)
        self.y_pred = self.model.predict(self.X_test)
        self.acc_score = accuracy_score(y_pred=self.y_pred, y_true=self.y_test)
        self.bal_acc_score = balanced_accuracy_score(y_pred=self.y_pred, y_true=self.y_test)
        self.conf_matrix = confusion_matrix(y_pred=self.y_pred, y_true=self.y_test)   

In [ ]:
# Adds the results stored in the evaluation class to the final results datatable.
def add_eval_to_final_results(eval, alg_name):
    
    n = len(final_results)
    
    final_results.loc[n, "algorithm"] = alg_name
    final_results.loc[n, "best_params"] = str(eval.best_param)
    final_results.loc[n, "accuracy"] = eval.acc_score
    final_results.loc[n, "balanced_accuracy"] = eval.bal_acc_score
    final_results.loc[n, "confusion_matrix"] = eval.conf_matrix

In [ ]:
# Defining a function that plots confusion matrixes of all three algorithms with respective parameters and model performance metrics

def plot_cm(model_number):

    ConfusionMatrixDisplay(confusion_matrix = final_results["confusion_matrix"][model_number], display_labels=eval_rf.clf.classes_).plot()
    
    plt.title("Experimental Setup:\n {}\n {}\n\n Accuracy: {}\n Balanced Accuracy: {}"\
                .format(final_results["algorithm"][model_number],
                        final_results["best_params"][model_number],
                        round(final_results["accuracy"][model_number],2),
                        round(final_results["balanced_accuracy"][model_number],2)))

## For adding new features

In [ ]:
def custom_window_function(df, sort_col, sort_asc, group_col, target_col, target_agg, lag, new_col_name):
    output = df\
                .sort_values(by=sort_col, ascending=sort_asc)\
                .set_index("index_id")\
                .groupby(group_col)\
                [[target_col]]\
                .rolling(lag)\
                .agg(target_agg)\
                .groupby(group_col).shift(1)\
                .reset_index()\
                .rename({target_col:new_col_name},axis=1)\
                .drop(group_col, axis=1)\
                .merge(df, on="index_id", how="inner")
    
    return output

In [ ]:
def multi_window_functions(df, target_agg, target_cols, lags):
    
    '''
    df
    target_agg: aggregation function as string
    target_cols: list with one home stats column and one away stats column
    lags: list with needed lags as integers
    '''
    
    group_col = "HomeTeam"
    target_col = target_cols[0]

    for lag in lags:
        df = custom_window_function(df=df, sort_col="Date", sort_asc=True, group_col=group_col,
                                        target_col=target_col, target_agg=target_agg, lag=lag,
                                         new_col_name="{}_{}_{}".format(target_agg, lag, target_col))


    group_col = "AwayTeam"
    target_col = target_cols[1]

    for lag in lags:
        df = custom_window_function(df=df, sort_col="Date", sort_asc=True, group_col=group_col,
                                        target_col=target_col, target_agg=target_agg, lag=lag,
                                         new_col_name="{}_{}_{}".format(target_agg, lag, target_col))
    
    return df

# Load Data

I have choosen a dataset that is presented on Kaggle:

https://www.kaggle.com/sashchernuh/european-football

It contains some statistics of a game with a lot of various betting odds from various companies. I want to use both betting odds (as pre-game information) and match statistics (as post-game information) as features for the machine learning models.

I downloaded the data directly from Kaggle. The data is stored in a sqlite-file. So I'm using the library "sqlite3" to load the data as a pandas datframe.

In [ ]:
sqlite3_conn = sqlite3.connect('../input/european-football/database.sqlite')

In [ ]:
tables = tables_in_sqlite_db(sqlite3_conn)

In [ ]:
tables

There are two tables in the dataset that represent the source where the data was gathered from by the kaggle author: "betfront" and "football_data" from football-data.co.uk.

In [ ]:
# Saving the raw data in a dictionary as pandas dataframes

df_dict = {}

for table in tables:
    df_dict[table] = pd.read_sql_query("SELECT * FROM '{}'".format(table), sqlite3_conn)

# Data Examination

In [ ]:
betfront = df_dict["betfront"]

In [ ]:
fd = df_dict["football_data"]

There seems to be a lot of information in the football_data dataframe. Let's focus on this dataset and take a glimpse look into the data.

In [ ]:
len(fd.columns)

Firstly, I want to check how many data we have in the dataset per Country, League and Season.

In [ ]:
plot_share_of_data(fd, "Season")

In [ ]:
plot_share_of_data(fd, "Country")

In [ ]:
plot_share_of_data(fd, "League")

We can see that the data is distributed unequal across the various season, leagues, and countries. Let's take a look on for which seasons which leagues have how much data.

In [ ]:
fd["Country_League"] = fd["Country"] + "_" + fd["League"]

In [ ]:
data = fd.value_counts(["Season", "Country_League"], sort=False, normalize=True).unstack().T

fig, ax = plt.subplots(figsize=(15,15))

sns.heatmap(data, annot = False, cbar_kws={'label': 'Share of total dataset (in percent)'})

Now, the picture is much clearer. There are Leagues, for which we have only data since 2012. Also, for this leagues, the season is labeled differently than for the other leagues. Also, the different leagues have a different share of data per season.

Me as a football fan, I know mostly the top 5 Leagues: German Bundesliga, English Premier League, Spanish La Liga, Italian Serie A and French Ligue 1. To make the data more compact and to be able to interpret the data more thoroughly, I will choose only observations from these 5 Leagues.

In [ ]:
top_5_leagues_list = ["Germany_Bundesliga 1", "England_Premier League", "Spain_La Liga Primera Division", "Italy_Serie A", "France_Le Championnat"]

fd_top_5 = fd[fd["Country_League"].isin(top_5_leagues_list)]

In [ ]:
data = fd_top_5.value_counts(["Season", "Country_League"], sort=False, normalize=True).unstack().T

fig, ax = plt.subplots(figsize=(7,7))
sns.heatmap(data, annot = False, cbar_kws={'label': 'Share of total dataset (in percent)'})

This looks much better. We can observe certain things:
- The German Bundesliga has systematically less data than the other leagues. This makes sense because in this league, there are only 18 and not 20 teams as in the other leagues.
- In 2020, France seems to have a gap. This gap is due to the Corona-Crisis – the French Ligue 1 (here called "Le Championnat") was the only league that cancelled the 2020-season due to corona.
- Before the season 2005/06, there are different inconsistent shares of data. We could investigate what happenned there, or we could just drop these observations. As we will see, we have still enough data to proceed on.

If needed, we can add this data and also other leagues, etc. later on back to the final dataset. But for now, I will skip them for the sake of data clearance and lean data.

In [ ]:
# Keep only the starting_year of the season for better clarity
fd_top_5['Season'] = fd_top_5['Season'].str[:4].astype(int)

In [ ]:
fd_top_5_datacut = fd_top_5[fd_top_5["Season"] > 2004]

In [ ]:
data = fd_top_5_datacut.value_counts(["Season", "Country_League"], sort=False, normalize=True).unstack().T

fig, ax = plt.subplots(figsize=(7,7))
sns.heatmap(data, annot = False, cbar_kws={'label': 'Share of total dataset (in percent)'})

Did we cut off maybe too much data?

In [ ]:
print("full data:", fd.shape[0], "matches.")
print("only top 5 leagues:", fd_top_5.shape[0], "matches.")
print("only top 5 leagues and seasons since 2005:", fd_top_5_datacut.shape[0], "matches.")

From my perspective, we still have enough data to proceed for the later prediction tasks. Of course we lost data, but we can add it later on again if necessary and right now, we have a better data clarity.

Now comes the heaviest part. We have 173 columns in the dataset. Firstly, we have to choose which columns we want to keep and secondly, if and how we should transform these columns. Also we should think about if and what kind of features we want to create newly. From http://www.football-data.co.uk/notes.txt, we have the following information about all columns:

Key to results data:

- Div = League Division
- Date = Match Date (dd/mm/yy)
- Time = Time of match kick off
- HomeTeam = Home Team
- AwayTeam = Away Team
- FTHG and HG = Full Time Home Team Goals
- FTAG and AG = Full Time Away Team Goals
- FTR and Res = Full Time Result (H=Home Win, D=Draw, A=Away Win)
- HTHG = Half Time Home Team Goals
- HTAG = Half Time Away Team Goals
- HTR = Half Time Result (H=Home Win, D=Draw, A=Away Win)

Match Statistics (where available)
- Attendance = Crowd Attendance
- Referee = Match Referee
- HS = Home Team Shots
- AS = Away Team Shots
- HST = Home Team Shots on Target
- AST = Away Team Shots on Target
- HHW = Home Team Hit Woodwork
- AHW = Away Team Hit Woodwork
- HC = Home Team Corners
- AC = Away Team Corners
- HF = Home Team Fouls Committed
- AF = Away Team Fouls Committed
- HFKC = Home Team Free Kicks Conceded
- AFKC = Away Team Free Kicks Conceded
- Boss = Home Team Offsides
- AO = Away Team Offsides
- HY = Home Team Yellow Cards
- AY = Away Team Yellow Cards
- HR = Home Team Red Cards
- AR = Away Team Red Cards
- HBP = Home Team Bookings Points (10 = yellow, 25 = red)
- ABP = Away Team Bookings Points (10 = yellow, 25 = red)

Note that Free Kicks Conceeded includes fouls, offsides and any other offense commmitted and will always be equal to or higher than the number of fouls. Fouls make up the vast majority of Free Kicks Conceded. Free Kicks Conceded are shown when specific data on Fouls are not available (France 2nd, Belgium 1st and Greece 1st divisions).

Note also that English and Scottish yellow cards do not include the initial yellow card when a second is shown to a player converting it into a red, but this is included as a yellow (plus red) for European games.


Key to 1X2 (match) betting odds data:

- B365H = Bet365 home win odds
- B365D = Bet365 draw odds
- B365A = Bet365 away win odds
- BSH = Blue Square home win odds
- BSD = Blue Square draw odds
- BSA = Blue Square away win odds
- BWH = Bet&Win home win odds
- BWD = Bet&Win draw odds
- BWA = Bet&Win away win odds
- GBH = Gamebookers home win odds
- GBD = Gamebookers draw odds
- GBA = Gamebookers away win odds
- IWH = Interwetten home win odds
- IWD = Interwetten draw odds
- IWA = Interwetten away win odds
- LBH = Ladbrokes home win odds
- LBD = Ladbrokes draw odds
- LBA = Ladbrokes away win odds
- PSH and PH = Pinnacle home win odds
- PSD and PD = Pinnacle draw odds
- PSA and PA = Pinnacle away win odds
- SOH = Sporting Odds home win odds
- SOD = Sporting Odds draw odds
- SOA = Sporting Odds away win odds
- SBH = Sportingbet home win odds
- SBD = Sportingbet draw odds
- SBA = Sportingbet away win odds
- SJH = Stan James home win odds
- SJD = Stan James draw odds
- SJA = Stan James away win odds
- SYH = Stanleybet home win odds
- SYD = Stanleybet draw odds
- SYA = Stanleybet away win odds
- VCH = VC Bet home win odds
- VCD = VC Bet draw odds
- VCA = VC Bet away win odds
- WHH = William Hill home win odds
- WHD = William Hill draw odds
- WHA = William Hill away win odds

Bb1X2 = Number of BetBrain bookmakers used to calculate match odds averages and maximums
- BbMxH = Betbrain maximum home win odds
- BbAvH = Betbrain average home win odds
- BbMxD = Betbrain maximum draw odds
- BbAvD = Betbrain average draw win odds
- BbMxA = Betbrain maximum away win odds
- BbAvA = Betbrain average away win odds

- MaxH = Market maximum home win odds
- MaxD = Market maximum draw win odds
- MaxA = Market maximum away win odds
- AvgH = Market average home win odds
- AvgD = Market average draw win odds
- AvgA = Market average away win odds



Key to total goals betting odds:

- BbOU = Number of BetBrain bookmakers used to calculate over/under 2.5 goals (total goals) averages and maximums
- BbMx>2.5 = Betbrain maximum over 2.5 goals
- BbAv>2.5 = Betbrain average over 2.5 goals
- BbMx<2.5 = Betbrain maximum under 2.5 goals
- BbAv<2.5 = Betbrain average under 2.5 goals

- GB>2.5 = Gamebookers over 2.5 goals
- GB<2.5 = Gamebookers under 2.5 goals
- B365>2.5 = Bet365 over 2.5 goals
- B365<2.5 = Bet365 under 2.5 goals
- P>2.5 = Pinnacle over 2.5 goals
- P<2.5 = Pinnacle under 2.5 goals
- Max>2.5 = Market maximum over 2.5 goals
- Max<2.5 = Market maximum under 2.5 goals
- Avg>2.5 = Market average over 2.5 goals
- Avg<2.5 = Market average under 2.5 goals



Key to Asian handicap betting odds:

- BbAH = Number of BetBrain bookmakers used to Asian handicap averages and maximums
- BbAHh = Betbrain size of handicap (home team)
- AHh = Market size of handicap (home team) (since 2019/2020)
- BbMxAHH = Betbrain maximum Asian handicap home team odds
- BbAvAHH = Betbrain average Asian handicap home team odds
- BbMxAHA = Betbrain maximum Asian handicap away team odds
- BbAvAHA = Betbrain average Asian handicap away team odds

- GBAHH = Gamebookers Asian handicap home team odds
- GBAHA = Gamebookers Asian handicap away team odds
- GBAH = Gamebookers size of handicap (home team)
- LBAHH = Ladbrokes Asian handicap home team odds
- LBAHA = Ladbrokes Asian handicap away team odds
- LBAH = Ladbrokes size of handicap (home team)
- B365AHH = Bet365 Asian handicap home team odds
- B365AHA = Bet365 Asian handicap away team odds
- B365AH = Bet365 size of handicap (home team)
- PAHH = Pinnacle Asian handicap home team odds
- PAHA = Pinnacle Asian handicap away team odds
- MaxAHH = Market maximum Asian handicap home team odds
- MaxAHA = Market maximum Asian handicap away team odds	
- AvgAHH = Market average Asian handicap home team odds
- AvgAHA = Market average Asian handicap away team odds



Closing odds (last odds before match starts)

As above but with an additional "C" character following the bookmaker abbreviation/Max/Avg

Betting odds for weekend games are collected Friday afternoons, and on Tuesday afternoons for midweek games.

Based on this information. we can cluster the existing columns by the following way:

1. Main Data ("Div" to "HTR")
2. Match Statistics("Attendance" to "ABP")
3. 1X2 Betting Odds ("B365H" to "AvgA")
4. Over/Under Betting Odds ("Bb0U" to "Avg<2.5")
5. Asian Handicap Betting Odds ("BbAH" to "AvgAHA")

In [ ]:
# Make a copy of the dataset that we want to continue with in the data preparation
df = fd_top_5_datacut.copy()

# Create an id column tha represents the index from the very first dataset "fd" (in case we need to relate somehow and also just to have an unique key-column per observation.)
df = df.reset_index().rename({"index":"index_id"},axis=1)

### Some Final Thoughts before Data Preparation:

We have a dataset with a lot of various information on a match basis. For every match, there is a home team and an away team with some respective statistics. Also, we have pre-game information (e.g. betting odds) that is available before a game and post-game information (e.g. amount of goals and other match statistics) that is only available after a game. The target variable for predictions is the outcome of the game (H for home win, D for draw and A for away win -> classification task).

# Data Preparation

For me, Data Preparation for classification tasks should be divided into the following parts:
1. Handling Missing Values
2. Handling Outliers
3. Dummyfication of categorical variables
4. Handling Imalanced Data
5. Partitioning into Train and Test Sets
6. Scaling Data

(Source: Deep Learning course of Prof. Stephan Schneider from FH Kiel.)

I will stick to the points 1-3 for every column cluster due to better overview.
Also for every column cluster, I will discuss which features could maybe dropped already because of unimportance.

I will not create new features firstly, but maybe later on.

The points 4-6 I will proceed finally on a joint dataframe.

## Column Cluster 1

In [ ]:
columns_1 = ["index_id", "Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR"]

In [ ]:
df_1 = df[columns_1]

In [ ]:
df_1

In [ ]:
df_1.dtypes

### Data Understanding

In [ ]:
# Plotting distribution of numerical columns
df_1.hist(layout = (7,6), figsize=(15,20))

plt.show()

The columns that show home and away half-time and full-time goals are right-skewed.

In [ ]:
categorical_features = ["Div", "FTR", "HTR"]

fig, ax = plt.subplots(nrows=len(categorical_features), ncols=1, figsize=(5,15))

for i, categorical_feature in enumerate(df_1[categorical_features]):
    df_1[categorical_feature].value_counts().plot(kind="barh", ax=ax[i]).set_title(categorical_feature)

fig.show()

# Code inspired by: https://stackoverflow.com/questions/31029560/plotting-categorical-data-with-pandas-and-matplotlib

At the Half Time, most games are still on Draw, whereas at Full-Time, most games are won by the home team.

### Handling Missing Values

In [ ]:
display_na_cols(df_1)

There are 3 Mising Values at the column HTR.

In [ ]:
df_1[df_1.isna().any(axis=1)]

In [ ]:
df_1.value_counts("HTHG", sort=False)

In [ ]:
df_1.value_counts("HTAG", sort=False)

We can see that the matches, where we have no half-time result (HTR) are also the only matches that have negative HTHG and HTAG. These three matches were interrupted due to fan riots, so I will delete them because the full-time result was not achieved by playing football but by disciplinary measures of the respective football federation.

In [ ]:
df_1_mv = df_1[~df_1.isna().any(axis=1)]

### Handling Outliers

To detect outliers, I will use the Tuckey-method. These outliers can be visualized by boxplotting the numerical columns.

In [ ]:
plt.figure()
df_1_mv.drop("index_id", axis=1).boxplot()

We can see some rare Outliers detected by Tuckey-method. Knowing this, I do not want to delete them, but keep them, because they represent real values and are not errors.

### Dealing with post-game information

The FTHG, FTAG, HTHG, and HTAG are post-game information. They cannot be used for predictions because this would cause the "leak-from-the-future" problem. But we can get aggregations of them of previous results. As very basic aggregation, the following function sorts the dataframe by date and gets the mean of the last x home or away goals. The resulting feature represents the home / away form of a team. Further Features will be included later, this one will be just as a basic feature to compare the model performance with new features later on.

In [ ]:
 df_1_mv_form = custom_window_function(df=df_1_mv, sort_col="Date", sort_asc=True, group_col="HomeTeam",
                                    target_col="FTHG", target_agg="mean", lag=3, new_col_name="Mean_3_FTHG")

 df_1_mv_form = custom_window_function(df=df_1_mv_form, sort_col="Date", sort_asc=True, group_col="AwayTeam",
                                    target_col="FTAG", target_agg="mean", lag=3, new_col_name="Mean_3_FTAG")

# Dropping Missing Values (the first matches of a team in this dataset that were used create the first mean but do not have a mean of the last x games because there is no data for the last x games)
df_1_mv_form = df_1_mv_form[~df_1_mv_form.isna().any(axis=1)]

After we have done the window operation, let's delete the goal-related columns (reason: the above mentioned "leak-from-the-future" problem).

In [ ]:
cols_leak = ["FTHG", "FTAG", "HTHG", "HTAG"]

df_1_leak = df_1_mv_form.drop(cols_leak, axis=1)

### Handling Categorical Variables

There are some categorical variables in df_1, but they are not usefull for the machine learning model, so I will delete them.

In [ ]:
cols_cat = ["Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTR", "HTR"]

df_1_cat = df_1_leak.drop(cols_cat, axis=1)

In [ ]:
df_1_final = df_1_cat.copy()

## Column Cluster 2

- Attendance = Crowd Attendance
- Referee = Match Referee
- HS = Home Team Shots
- AS = Away Team Shots
- HST = Home Team Shots on Target
- AST = Away Team Shots on Target
- HHW = Home Team Hit Woodwork
- AHW = Away Team Hit Woodwork
- HC = Home Team Corners
- AC = Away Team Corners
- HF = Home Team Fouls Committed
- AF = Away Team Fouls Committed
- HFKC = Home Team Free Kicks Conceded
- AFKC = Away Team Free Kicks Conceded
- Boss = Home Team Offsides
- AO = Away Team Offsides
- HY = Home Team Yellow Cards
- AY = Away Team Yellow Cards
- HR = Home Team Red Cards
- AR = Away Team Red Cards
- HBP = Home Team Bookings Points (10 = yellow, 25 = red)
- ABP = Away Team Bookings Points (10 = yellow, 25 = red)

These are all post-game statistics. As I want to include only betting odds and one basic feature for the first model, I will handle them later on.

In [ ]:
columns_2 = ["index_id", "Attendance", "Referee", "HS", "AS", "HST", "AST", "HHW", "AHW", "HC", "HF", "AF", "HFKC", "AFKC", "AO", "HY", "AY", "HR", "AR", "HBP", "ABP"]

In [ ]:
df_2 = df[columns_2]

### Data Understanding

In [ ]:
# Plotting distribution of numerical columns
df_2.hist(layout = (7,6), figsize=(15,20))

plt.show()

The distribution of the post-match statistics seem to also right-skewed. Despite of the Referree column that is empty, there is no categorical column.

### Handling Missing Values

In [ ]:
display_na_cols(df_2)

Some columns consist only out of Missing Values, so I will delete them.

In [ ]:
df_2_mv_1 = df_2.drop(["Attendance", "Referee", "HHW", "AHW", "HFKC", "AFKC", "AO", "HBP", "ABP"], axis=1)

In [ ]:
display_na_cols(df_2_mv_1)

In [ ]:
df_2_mv_1[df_2_mv_1[["HS", "AS"]].isna().any(axis=1)]

For many of the matches where HS and AS are missing, we have no information at all despite of information about Yellow (HY & AY) and Red Cards (HT & AR) which are not very important. So I will delete them, too.

In [ ]:
df_2_mv_2 = df_2_mv_1[~df_2_mv_1[["HS", "AS", "HY"]].isna().any(axis=1)]

In [ ]:
display_na_cols(df_2_mv_2)

In [ ]:
# Counting Missing Values per column and summing up per league and season
df_2_mv_2[df_2_mv_2[["HF", "AF", "HC", "HST", "AST"]].isna().any(axis=1)]\
    .merge(df[["index_id", "League", "Season"]], on="index_id", how="inner")\
    .groupby(["League", "Season"]).count()

In [ ]:
# Summing up the count of all observations that we have per League and Season where we have Missing Values
df[["index_id", "League", "Season"]]\
    [df.League.isin(["Bundesliga 1", "Le Championnat"]) & df.Season.between(2005, 2006, inclusive='both')]\
    .groupby(["League", "Season"]).count()

We can see that for all games in German Bundesliga in 2005/2006, we have no observations about the match stats despite of shots per target.

The same applies to the French Le Championnat for the seasons 2005/2006 & 2006/2007.

Sometimes Missing Values can be filled with the mean of the feature, but in this case, this does not make sense because all games in these seasons would just get the same result (or the same result per team). So I will delete these seasons in these leagues from the dataset, too. Before, I will replace the two missing values in Season 2011 with the mean of this season.

In [ ]:
# Save seasons mean values in dictionary
na_fill_values = df_2_mv_2\
    .merge(df[["index_id", "League", "Season"]]\
            [(df.Season == 2011) & (df.League == 'Le Championnat')],
            on="index_id", how="inner")[["HF", "AF"]].mean().to_dict()

In [ ]:
# Save relevant index_ids in list
na_index_ids = df_2_mv_2\
    .merge(df[["index_id", "League", "Season"]]\
            [(df.Season == 2011) & (df.League == 'Le Championnat') & (df.HF.isnull())],
            on="index_id", how="inner").index_id.tolist()

In [ ]:
df_2_mv_3 = df_2_mv_2.copy()

# Replace relevant index_ids values with the above resulted means
df_2_mv_3[df_2_mv_3.index_id.isin(na_index_ids)] = df_2_mv_3[df_2_mv_3.index_id.isin(na_index_ids)].fillna(value=na_fill_values)

In [ ]:
# Select all index_ids from the relevant League-Season combinations
drop_index_ids = df[((df.League == 'Le Championnat') & (df.Season.between(2005,2006))) | ((df.League == 'Bundesliga 1') & (df.Season == 2005))].index_id

In [ ]:
# Remove the relevant league-season combinations from the main df
df_cut = df[~df.index_id.isin(drop_index_ids)]

### Handling Outliers

In [ ]:
plt.figure()
df_2_mv_3.drop("index_id", axis=1).boxplot()

Again, as the outliers rely to real observations, that are just unusual in a typical match and are not measurement mistakes, I will keep them in the dataset as they are.

In [ ]:
df_2_final = df_2_mv_3.copy()

## Column Cluster 3

- B365H = Bet365 home win odds
- B365D = Bet365 draw odds
- B365A = Bet365 away win odds
- BSH = Blue Square home win odds
- BSD = Blue Square draw odds
- BSA = Blue Square away win odds
- BWH = Bet&Win home win odds
- BWD = Bet&Win draw odds
- BWA = Bet&Win away win odds
- GBH = Gamebookers home win odds
- GBD = Gamebookers draw odds
- GBA = Gamebookers away win odds
- IWH = Interwetten home win odds
- IWD = Interwetten draw odds
- IWA = Interwetten away win odds
- LBH = Ladbrokes home win odds
- LBD = Ladbrokes draw odds
- LBA = Ladbrokes away win odds
- PSH and PH = Pinnacle home win odds
- PSD and PD = Pinnacle draw odds
- PSA and PA = Pinnacle away win odds
- SOH = Sporting Odds home win odds
- SOD = Sporting Odds draw odds
- SOA = Sporting Odds away win odds
- SBH = Sportingbet home win odds
- SBD = Sportingbet draw odds
- SBA = Sportingbet away win odds
- SJH = Stan James home win odds
- SJD = Stan James draw odds
- SJA = Stan James away win odds
- SYH = Stanleybet home win odds
- SYD = Stanleybet draw odds
- SYA = Stanleybet away win odds
- VCH = VC Bet home win odds
- VCD = VC Bet draw odds
- VCA = VC Bet away win odds
- WHH = William Hill home win odds
- WHD = William Hill draw odds
- WHA = William Hill away win odds

- Bb1X2 = Number of BetBrain bookmakers used to calculate match odds averages and maximums
- BbMxH = Betbrain maximum home win odds
- BbAvH = Betbrain average home win odds
- BbMxD = Betbrain maximum draw odds
- BbAvD = Betbrain average draw win odds
- BbMxA = Betbrain maximum away win odds
- BbAvA = Betbrain average away win odds

- MaxH = Market maximum home win odds
- MaxD = Market maximum draw win odds
- MaxA = Market maximum away win odds
- AvgH = Market average home win odds
- AvgD = Market average draw win odds
- AvgA = Market average away win odds

Here we have a lot of columns, but they all just represent the HDA-odds (Home Win – Draw – Away Win) for every game by different betting operators. Luckily, we have also columns of bookmakers that show the average and maximum odds for every game so let's evaluate only these ones.

In [ ]:
columns_3 = ["index_id", "Bb1X2", "BbMxH", "BbAvH", "BbMxD", "BbAvD", "BbMxA", "BbAvA", "AvgH", "AvgD", "AvgA", "MaxH", "MaxD", "MaxA"]

df_3 = df_cut[columns_3]

### Data Understanding

In [ ]:
# Plotting distribution of numerical columns
df_3.hist(layout = (7,6), figsize=(15,20))

plt.show()

The odds are highly right-skewed. This makes sense because firstly, an odd has to be bigger than 0. Also, usually teams playing abilities in a league should be kind of in a similar range (of course, there are also big differences, but the differences are not as big as between a third and a first division team). This just should show that it makes sense that odds usually are in a "low" range between 1 and 3 e.g..

### Missing Values

In [ ]:
display_na_cols(df_3)

We have 3552 missing values in all columns and some columns have no values at all.


In [ ]:
# Counting Missing Values per column and summing up per league and season
df_3.merge(df[["index_id", "League", "Season"]], on="index_id", how="inner")\
    .groupby(["League", "Season"]).count().T

We have two different column sources: "Betbrain" and "Market". The Betbrain-columns count for all seasons until 2018 and the market-columns account for all columns after 2018. That's very good to see. I will create general columns which will be the sums of both column types.

In [ ]:
# Filling all NA-Values with 0 to be able to add columns with each other
df_3_na = df_3.fillna(0)

# Adding columns to new column so the new column does not contain na's
df_3_na["avg_h"] = df_3_na["AvgH"] + df_3_na["BbAvH"]
df_3_na["avg_d"] = df_3_na["AvgD"] + df_3_na["BbAvD"]
df_3_na["avg_a"] = df_3_na["AvgA"] + df_3_na["BbAvA"]

df_3_na["max_h"] = df_3_na["MaxH"] + df_3_na["BbMxH"]
df_3_na["max_d"] = df_3_na["MaxD"] + df_3_na["BbMxD"]
df_3_na["max_a"] = df_3_na["MaxA"] + df_3_na["BbMxA"]

# Select only relevant columns
df_3_na = df_3_na[["index_id", "avg_h", "avg_d", "avg_a", "max_h", "max_d", "max_a"]]

In [ ]:
df_3_final = df_3_na.copy()





As before, we will not handle outliers because these values represent real values.

There are no categorical columns that have to be proceeded.


## Column Cluster 4

- BbOU = Number of BetBrain bookmakers used to calculate over/under 2.5 goals (total goals) averages and maximums
- BbMx>2.5 = Betbrain maximum over 2.5 goals
- BbAv>2.5 = Betbrain average over 2.5 goals
- BbMx<2.5 = Betbrain maximum under 2.5 goals
- BbAv<2.5 = Betbrain average under 2.5 goals

- GB>2.5 = Gamebookers over 2.5 goals
- GB<2.5 = Gamebookers under 2.5 goals
- B365>2.5 = Bet365 over 2.5 goals
- B365<2.5 = Bet365 under 2.5 goals
- P>2.5 = Pinnacle over 2.5 goals
- P<2.5 = Pinnacle under 2.5 goals
- Max>2.5 = Market maximum over 2.5 goals
- Max<2.5 = Market maximum under 2.5 goals
- Avg>2.5 = Market average over 2.5 goals
- Avg<2.5 = Market average under 2.5 goals

The columns look pretty similar as in Cluster 3 despite ofthe fact that the information is a betting odd for under/over 2.5 goals and for an HDA-event.

In [ ]:
columns_4 = ["index_id", "BbOU", "BbMx>2.5", "BbAv>2.5", "BbMx<2.5", "BbAv<2.5", "GB>2.5", "GB<2.5", "B365>2.5", "B365<2.5", "P>2.5", "P<2.5", "Max>2.5", "Max<2.5", "Avg>2.5", "Avg<2.5"]

df_4 = df_cut[columns_4]

### Data Understanding

In [ ]:
# Plotting distribution of numerical columns
df_4.hist(layout = (7,6), figsize=(15,20))

plt.show()

### Handling Missing Values

In [ ]:
display_na_cols(df_4)

In [ ]:
# Counting Missing Values per column and summing up per league and season
df_4.merge(df[["index_id", "League", "Season"]], on="index_id", how="inner")\
    .groupby(["League", "Season"]).count().T

We have the same observation as in the cluster 3, so I will do the same procedure here. Also, the "GB"-columns are completely empty so I can drop them.

In [ ]:
# Filling all NA-Values with 0 to be able to add columns with each other
df_4_na = df_4.drop(["GB>2.5", "GB<2.5"], axis=1).fillna(0)

# Adding columns to new column so the new column does not contain na's
df_4_na["avg_over_2_5"] = df_4_na["BbAv>2.5"] + df_4_na["Avg>2.5"]
df_4_na["avg_under_2_5"] = df_4_na["BbAv<2.5"] + df_4_na["Avg<2.5"]

df_4_na["max_over_2_5"] = df_4_na["BbMx>2.5"] + df_4_na["Max>2.5"]
df_4_na["max_under_2_5"] = df_4_na["BbMx<2.5"] + df_4_na["Max<2.5"]

# Select only relevant columns
df_4_na = df_4_na[["index_id", "avg_over_2_5", "avg_under_2_5", "max_over_2_5", "max_under_2_5"]]

In [ ]:
df_4_final = df_4_na.copy()

## Column Cluster 5

 For a better understanding on how Asian Handicap Odds work, please read the following website: https://www.betshoot.com/betting-guides/asian-handicap-betting/

- BbAH = Number of BetBrain bookmakers used to Asian handicap averages and maximums
- BbAHh = Betbrain size of handicap (home team)
- AHh = Market size of handicap (home team) (since 2019/2020)
- BbMxAHH = Betbrain maximum Asian handicap home team odds
- BbAvAHH = Betbrain average Asian handicap home team odds
- BbMxAHA = Betbrain maximum Asian handicap away team odds
- BbAvAHA = Betbrain average Asian handicap away team odds

- GBAHH = Gamebookers Asian handicap home team odds
- GBAHA = Gamebookers Asian handicap away team odds
- GBAH = Gamebookers size of handicap (home team)
- LBAHH = Ladbrokes Asian handicap home team odds
- LBAHA = Ladbrokes Asian handicap away team odds
- LBAH = Ladbrokes size of handicap (home team)
- B365AHH = Bet365 Asian handicap home team odds
- B365AHA = Bet365 Asian handicap away team odds
- B365AH = Bet365 size of handicap (home team)
- PAHH = Pinnacle Asian handicap home team odds
- PAHA = Pinnacle Asian handicap away team odds
- MaxAHH = Market maximum Asian handicap home team odds
- MaxAHA = Market maximum Asian handicap away team odds	
- AvgAHH = Market average Asian handicap home team odds
- AvgAHA = Market average Asian handicap away team odds

As in Column Clusters 3 and 4, we have some individual odds of various bet brokers and also market averages, which I will choose for further investigation.

In [ ]:
columns_5 = ["index_id","BbMxAHH", "BbAvAHH", "BbMxAHA", "BbAvAHA", "MaxAHH", "MaxAHA", "AvgAHH", "AvgAHA"]

df_5 = df_cut[columns_5]

### Data Understanding

In [ ]:
# Plotting distribution of numerical columns
df_5.hist(layout = (7,6), figsize=(15,20))

plt.show()

### Handling Missing Values

In [ ]:
display_na_cols(df_5)

In [ ]:
# Counting Missing Values per column and summing up per league and season
df_5.merge(df[["index_id", "League", "Season"]], on="index_id", how="inner")\
    .groupby(["League", "Season"]).count().T

As in the column clusters 4 and 5, we have the same observation regarding the structure of the available odds information. So we will perform also here the same operations.

In [ ]:
# Filling all NA-Values with 0 to be able to add columns with each other
df_5_na = df_5.fillna(0)

# Adding columns to new column so the new column does not contain na's
df_5_na["avg_ah_h"] = df_5_na["AvgAHH"] + df_5_na["BbAvAHH"]
df_5_na["avg_ah_a"] = df_5_na["AvgAHA"] + df_5_na["BbAvAHA"]

df_5_na["max_ah_h"] = df_5_na["MaxAHH"] + df_5_na["BbMxAHH"]
df_5_na["max_ah_a"] = df_5_na["MaxAHA"] + df_5_na["BbMxAHA"]

# Select only relevant columns
df_5_na = df_5_na[["index_id", "avg_ah_h", "avg_ah_a", "max_ah_h", "max_ah_a"]]

In [ ]:
df_5_final = df_5_na.copy()

## Prepare Final df to continue Data Prep

In [ ]:
# Merging all single proceeded df's
df_prep = df_5_final.merge(df_4_final, on="index_id", how="inner")\
                    .merge(df_3_final, on="index_id", how="inner")\
                    .merge(df_1_final, on="index_id", how="inner")

# Merging with df by index_id to get the column "FTR" taht represents the full time match result and will serve as the target variable
df_prep = df_prep.merge(df[["index_id", "FTR"]], on="index_id", how="left")

In [ ]:
df_prep.info()

In [ ]:
data = df_prep[["avg_h", "avg_d", "avg_a", "Mean_3_FTHG", "Mean_3_FTAG", "FTR"]]
corr = data.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

sns.pairplot(data,  hue="FTR", markers=["o", "s", "D"], palette="Set2", corner=True)

plt.show()

In this scatter plots, we can see some interesting patterns.
- E.g.: If, for one match, an average draw odd (avg_d) is higher than 5 and an average home odd is lower than appr. 3 (avg_h), then it's very likely that the match outcome is a Home Win (H).
- Looking at the mean goals of the last 3 home/away matches, it is also possible to see visually such paterns that lead to a certain match outcome.
- It is interesting to note that it is not possible to see visually a pattern for a Draw event (D) in any of the scatter plots. The Draw events seem to be distributed at the first glimpse almost totally random throughout the scatter plots.
- Comparing the histograms of the mean last home goals (Mean_3_FTHG) and mean last away goals (Mean_3_FTAG), it is interesting to note that the curve of the mean home goals is much smoother than of the away goals what can lead to an assumption that home goals could be more predictable than away goals if we would do a regression task.

## Feature Selection

Firstly, I will include all features in the models. Later we will perform feature selection methods like PCA and univariate correlation to try to optimize the models.

## Handling Imbalanced Data

In [ ]:
df_prep["FTR"].value_counts(normalize=True).plot(kind="bar")

plt.xlabel("Match Results")
plt.ylabel("Share in %")

We can see that the target variable is distributed unequally. There are much more home wins than away wins. This is actually one reason why it is so important to have all the games statistics divided into home and away statistics. But there is no class that is occuring heavily much more/less than the others so I will not balance the data firstly.

## Partitioning into Training and Test Sets

In [ ]:
# Splitting Input features to X and target feature to y
X, y = df_prep.set_index("index_id").drop("FTR", axis=1), df_prep["FTR"]

In [ ]:
# Splitting X and y to train and test sets with a distribution of 70 / 30 and choosing random seed as 42.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True, stratify=y)

## Scaling Data

For some Machine Learning Methods, scaling data is a requirement, because they are sensitive to the the variance in the data.

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(X_train)

X_train_scaled = scaler_fitted.transform(X_train)
X_test_scaled = scaler_fitted.transform(X_test)

## Prepare taget feature

Not needed because sklearn does that automatically.

# Modelling

## Benchmark Model

- The typical basic benchmark model for football predictions is to say that always the home team wins as this is the case in appr. 47% of all matches. 
- I will two metrics for this basic model – accuracy and balanced accuracy. As the question of this analysis is to try to predict a match outcome to perfrom betting based on the outcome, the accuracy metric is the right metric to choose. The reason is that for betting, I want to achieve the highest possible amount of right predictions. The amount of right predictions out of all predictions is exactly what the accuracy describes, so this is the metric that should be optimized in the modelling process.

In [ ]:
# Creating a prediction that predicts only the Home Win event for all matches.
y_pred_basic = pd.Series("H", index=range(len(y_test)))

In [ ]:
# Performing the evaluation of the basic bencmark "model"
print("accuracy score: ", round(accuracy_score(y_test, y_pred_basic),4))
print("balanced accuracy score: ", round(balanced_accuracy_score(y_test, y_pred_basic),4))

print(classification_report(y_test, y_pred_basic, zero_division=1))

- The basic model predicts roundabout 47 % of all match results right. This is not bad.
- The recall for home win is perfect because we can predict of course all actual home wins correctly. But one problem of this model is that the recall of the other outcomes is 0 what means that no events of the other two classes are predicted correctly.
- So Let's try now to create models that are better than the basic model.

## Run Machine Learning Experiment

Firstly, I created a machine learning setup that will proceed a Stratified KFold Cross Validation and Grid Search in chapter 3.4.

- Now I will perform the modelling setup for various algorithms and create respectively hyperparameter grids in which to search for the best parameter combination.
- The parameter grids are created by my personal assumptions of right parameter ranges and also under the restriction of computational limits of my computer.
- So I am aware of that there might be some other hyperparameter combinations that give better results, but because of computational limits I will not be able to find the best possible combinations. But, as often in data science, hyperparameter tuning is not the biggest lever to increase models performance to a high extend.

## Decision Tree

In [ ]:
# Create the parameter grid
param_grid = {
    'max_depth': [1,2,3,5,7,9,12,15,20,None],    
}

# Create the estimator
estimator_dt = DecisionTreeClassifier(criterion="gini", random_state=42)

# Perform the experimental setup on the above defined estimator and param grid
cv_results_decision_tree = cv_procedure(estimator=estimator_dt, param_grid=param_grid, X=X_train, y=y_train)

In [ ]:
cv_results_decision_tree

It is interesting to see that the best results were achieved with small trees. That shows that there are not many parameters that have the biggest contribution on the model.

## Random Forest

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [1, 3, 5, 10, None]
    }

# Create the estimator
estimator_rf = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest = cv_procedure(estimator=estimator_rf, param_grid=param_grid, X=X_train, y=y_train)

In [ ]:
cv_results_random_forest

Logically, also in the Random Forest model, the better models are not that deep as the worse ones. With increasing number of estimators, the accuracy also increases, but in a really small amount so due to computational limits, it is not recomendable to increasethe amount of trees in the random forest algorithm.

## KNN

In [ ]:
# Create the parameter grid
param_grid = {
    'n_neighbors': [5, 50, 100, 250, 500, 750, 1000, 1500]
    }

# Create the estimator
estimator_knn = KNeighborsClassifier()

# Perform the experimental setup
cv_results_knn = cv_procedure(estimator=estimator_knn, param_grid=param_grid, X=X_train_scaled, y=y_train)

In [ ]:
cv_results_knn

## Logistic Regression

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_scaled, y=y_train)

In [ ]:
cv_results_lr

## Gradient Boosting

In [ ]:
# Create the parameter grid
param_grid = {
    #'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, 15],
    "learning_rate":[0.1,1,10]
    }

# Create the estimator
estimator_gb = GradientBoostingClassifier(random_state=42)

# Perform the experimental setup
cv_results_gb = cv_procedure(estimator=estimator_gb, param_grid=param_grid, X=X_train_scaled, y=y_train)

In [ ]:
cv_results_gb

# Evaluation

Now I will use the class "evaluation" that I created in chapter 3.5 and create such an object for each algorithm to be able to compare them at the end.

## Evaluation Preparation

In [ ]:
eval_dt = evaluation(cv_results_decision_tree, DecisionTreeClassifier, estimator_dt, X_train, y_train, X_test, y_test)
eval_dt.evaluate()

In [ ]:
eval_rf = evaluation(cv_results_random_forest, RandomForestClassifier, estimator_rf, X_train, y_train, X_test, y_test)
eval_rf.evaluate()

In [ ]:
eval_knn = evaluation(cv_results_knn, KNeighborsClassifier, estimator_knn, X_train_scaled, y_train, X_test_scaled, y_test)
eval_knn.evaluate()

In [ ]:
eval_lr = evaluation(cv_results_lr, LogisticRegression, estimator_lr, X_train_scaled, y_train, X_test_scaled, y_test)
eval_lr.evaluate()

In [ ]:
eval_gb = evaluation(cv_results_gb, GradientBoostingClassifier, estimator_gb, X_train_scaled, y_train, X_test_scaled, y_test)
eval_gb.evaluate()

## Show Final Results

In [ ]:
# Create an empty pandas dataframe with evaluation metrics
final_results = pd.DataFrame(columns=["algorithm", "best_params", "accuracy", "balanced_accuracy", "confusion_matrix"])

In [ ]:
evals = [eval_dt, eval_rf, eval_knn, eval_lr, eval_gb]
names = ["DT","RF", "KNN", "LR", "GB"]

for i in range(len(evals)):
    add_eval_to_final_results(evals[i], names[i])

In [ ]:
final_results.sort_values("accuracy", ascending=False)

We can see that all the algorithms almost perform equally around a value of appr. 53 % accuracy. This is an interesting observation: none of the algorithms is surely the best one for this classification task, all give more or less a similar result. Also, we can see that the balanced accuracy is always lower for around 9 % points than accuracy – an indication that we should probably balance the data.

## Confusion Matrices

In [ ]:
for i in range(len(evals)):
    plot_cm(model_number=i)

The unbalanced nature of the data is becoming even more obvious by looking at the confusion matrices – the D events almost never are predicted by any of the algorithm. For every algorithm, most of the predictions are H – they all tend to predict the major class. My assumpotion, that we do not have to balance the data, was maybe wrong and we maybe do have to balance the data. I will do it in chapter 9. Contrarly to the previous Modelling chapter 7, I will not do it with all algorithms – we already saw that there is no big difference between them. I will just choose the RandomForest and Logistic Regression algorithms as they belong to the best 3 ones (although Gradient Boosting is the second best, I will skip using this one further because of its very long computational time).

In [ ]:
#test = final_results["confusion_matrix"][0]

# Model with balanced data

## Sample Data

I will use two sampling options – Oversampling and SMOTE. Oversampling adds randomly choosen instances of the minority classes to the dataset while SMOTE is an oversampling technique where synthetic samples are generated for the minority class. This algorithm helps to overcome the overfitting problem posed by random oversampling. SMOTE works by selecting examples that are close in the feature space, drawing a line between the examples in the feature space and drawing a new sample at a point along that line. Specifically, a random example from the minority class is first chosen.

Source: https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/

In [ ]:
# Define theoversampling object
ros = RandomOverSampler(random_state=42)

# Perform the Oversampling process with the oversampling object on the train input and target features
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)
X_train_scaled_ros, y_train_scaled_ros = ros.fit_resample(X_train_scaled, y_train)

# Print the distribution of the balanced and unbalanced target variables
print('Original train target variable distribution:\n{}'.format(y_train.value_counts()))
print('Oversampled train target variable distribution:\n{}'.format(y_train_ros.value_counts()))

# The code above was inspired copied from:
# https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.RandomOverSampler.html

In [ ]:
# Define the smote object
smote = SMOTE(random_state=42)

# Perform the smote process with the undersampling object on the train input and target features
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
X_train_scaled_smote, y_train_scaled_smote = smote.fit_resample(X_train_scaled, y_train)

# Print the distribution of the balanced and unbalanced target variables
print('Original train target variable distribution:\n{}'.format(y_train.value_counts()))
print('Oversampled train target variable distribution:\n{}'.format(y_train_smote.value_counts()))

## Perform Modelling on sample data

To implement oversampling in cross validation, there is an important step that has to taken into consideration. Every train fold, that is created in the Cross Validation, has to be oversampled, while its test fold should not be oversampled. To achieve this, I created a new function that considers a pipe function from the package imblearn from sklearn. I implemented this in the function cv_procedure_sample.

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [3, 5, 10]
    }

# Perform the experimental setup
cv_results_random_forest_ros = cv_procedure_sample(estimator=estimator_rf, param_grid=param_grid, X=X_train_ros, y=y_train_ros, sampler=ros, alg_name="RF_ros", name_add='randomforestclassifier__', scoring="accuracy")

In [ ]:
final_results.loc[len(final_results)] = cv_results_random_forest_ros

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Perform the experimental setup
cv_results_lr_ros = cv_procedure_sample(estimator=estimator_lr, param_grid=param_grid, X=X_train_scaled_ros, y=y_train_scaled_ros, sampler=ros, alg_name="LR_ros", name_add='logisticregression__', scoring="accuracy")

In [ ]:
final_results.loc[len(final_results)] = cv_results_lr_ros

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [3, 5, 10]
    }

# Perform the experimental setup
cv_results_random_forest_smote = cv_procedure_sample(estimator=estimator_rf, param_grid=param_grid, X=X_train_smote, y=y_train_smote, sampler=smote, alg_name="RF_smote", name_add='randomforestclassifier__', scoring="accuracy")

In [ ]:
final_results.loc[len(final_results)] = cv_results_random_forest_smote

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Perform the experimental setup
cv_results_lr_smote = cv_procedure_sample(estimator=estimator_lr, param_grid=param_grid, X=X_train_scaled_smote, y=y_train_scaled_smote, sampler=smote, alg_name="LR_smote", name_add='logisticregression__', scoring="accuracy")

In [ ]:
final_results.loc[len(final_results)] = cv_results_lr_smote

## Evaluate

In [ ]:
final_results.sort_values("accuracy", ascending=False)

This is a very interesting observation.
- The accuracy of the models with oversampled data sinks.
- By taking oversampled data for model training, we definetly got rid of the gap between accuracy and balanced_accuracy score.
- BUT: As accuracy is our main metric because we want to use the prediction results for betting, oversampling does not help us.
- For our purposes, we would stick rather to models which predictions maybe are not distributed as the target feature but which overall give us a better result in the metric that we want to achieve.
- In other use cases, deciding for sticking to the procedure of sampling may be the right choice, but in this case it is not.

## Confusion Matrices

In [ ]:
for i in [1,2,6,7,8]:
    plot_cm(model_number=i)

We can clearly see by looking at the above confusion matrices what causes the gap between the accuracy score and the balanced accuracy score of models without oversampling.
- Models that were trained with sampled data do predict the draw event, while models that were trained without sampled data do not predict the draw event almost at all.
- This is an interesting observation and fits our observation in chapter 6.6 where we saw that we could not find visual patterns in the scatter plots for the draw event, but for the Home and Away win event, we could identify visually patterns in the data.
- Oversampling therefore let the model predict generally more Draw events, what actually is a good thing. BUT: from the predicted draw events, there are too many that are predicted wrongly. In fact, the models just minimally predict more than 1/3 of the predicted draw events correctly (1/3 would be random choice).
- As the accuracy metric is better without using oversampled data, I will not continue further with oversampling.

The question is: How can we train the model to receive a higher accuracy? Maybe we can achieve that by giving the model more information so it can better predict the draw outcome. If we achieve this, the model will get automatically better.

# Models with new Features

- I will create various features out of the post-information statistics from column clusters 1 & 2.
- The new features basically are created to represent the short- (3 games), middle- (10 games) and long-term (30 games) form of every team regarding the various statistics.
- I created in chapter 3.6 the needed functions to create these features. They are basically window functions.

## Creating new Features

### Column Cluster 1

In [ ]:
df_1_nf = df_1_mv.copy()

df_1_nf["FTR_H"] = df_1_nf["FTR"].replace({"H":3, "D":1, "A":0})
df_1_nf["FTR_A"] = df_1_nf["FTR"].replace({"H":0, "D":1, "A":3})

df_1_nf["HTR_H"] = df_1_nf["HTR"].replace({"H":3, "D":1, "A":0})
df_1_nf["HTR_A"] = df_1_nf["HTR"].replace({"H":0, "D":1, "A":3})

#### FTHG & FTAG

In [ ]:
target_agg = "mean"
target_cols = ["FTHG", "FTAG"]
lags = [3, 10, 30]

In [ ]:
df_1_nf = multi_window_functions(df = df_1_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HTHG & HTAG

In [ ]:
target_agg = "mean"
target_cols = ["HTHG", "HTAG"]
lags = [3, 10, 30]

In [ ]:
df_1_nf = multi_window_functions(df = df_1_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### FTR_H & FTR_A

In [ ]:
target_agg = "sum"
target_cols = ["FTR_H", "FTR_A"]
lags = [3, 10, 30]

In [ ]:
df_1_nf = multi_window_functions(df = df_1_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HTR_H & HTR_A

In [ ]:
target_agg = "sum"
target_cols = ["HTR_H", "HTR_A"]
lags = [3, 10, 30]

In [ ]:
df_1_nf = multi_window_functions(df = df_1_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### Finalize

In [ ]:
# Dropping Missing Values (the first matches of a team in this dataset that were used create the first mean but do not have a mean of the last x games because there is no data for the last x games)
df_1_nf_final = df_1_nf[~df_1_nf.isna().any(axis=1)]

In [ ]:
cols_leak = ["FTHG", "FTAG", "HTHG", "HTAG", "FTR_H", "FTR_A", "HTR_H", "HTR_A"]

df_1_nf_final = df_1_nf_final.drop(cols_leak, axis=1)

In [ ]:
cols_cat = ["Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTR", "HTR"]

df_1_nf_final = df_1_nf_final.drop(cols_cat, axis=1)

In [ ]:
df_1_nf_final.info()

### Column Cluster 2

In [ ]:
df_2_nf = df_2_final.copy().merge(df[["index_id", "Date", "HomeTeam", "AwayTeam"]], on="index_id", how="inner")

#### HS & AS

In [ ]:
target_agg = "mean"
target_cols = ["HS", "AS"]
lags = [3, 10, 30]

In [ ]:
df_2_nf = multi_window_functions(df = df_2_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HST & AST

In [ ]:
target_agg = "mean"
target_cols = ["HST", "AST"]
lags = [3, 10, 30]

In [ ]:
df_2_nf = multi_window_functions(df = df_2_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HF & AF

In [ ]:
target_agg = "mean"
target_cols = ["HF", "AF"]
lags = [3, 10, 30]

In [ ]:
df_2_nf = multi_window_functions(df = df_2_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HY & AY

In [ ]:
target_agg = "mean"
target_cols = ["HY", "AY"]
lags = [3, 10, 30]

In [ ]:
df_2_nf = multi_window_functions(df = df_2_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### HR & AR

In [ ]:
target_agg = "mean"
target_cols = ["HR", "AR"]
lags = [3, 10, 30]

In [ ]:
df_2_nf = multi_window_functions(df = df_2_nf,
                                   target_agg = target_agg,
                                   target_cols = target_cols,
                                   lags = lags)

#### Finalize

In [ ]:
# Dropping Missing Values (the first matches of a team in this dataset that were used create the first mean but do not have a mean of the last x games because there is no data for the last x games)
df_2_nf_final = df_2_nf[~df_2_nf.isna().any(axis=1)]

In [ ]:
cols_leak = ["HS", "AS", "HST", "AST", "HC", "HF", "AF", "HY", "AY", "HR", "AR"]

df_2_nf_final = df_2_nf_final.drop(cols_leak, axis=1)

In [ ]:
cols_cat = ["Date", "HomeTeam", "AwayTeam"]

df_2_nf_final = df_2_nf_final.drop(cols_cat, axis=1)

## Data Prep for Modelling with new features

### Prepare Final Dataprep

Now I will join the new features (df_2_nf_final & df_1_nf_final) with the features from the other column clusters that are related to odds.

In [ ]:
# Merging all single proceeded df's
df_prep_nf = df_5_final.merge(df_4_final, on="index_id", how="inner")\
                       .merge(df_3_final, on="index_id", how="inner")\
                       .merge(df_2_nf_final, on="index_id", how="inner")\
                       .merge(df_1_nf_final, on="index_id", how="inner")

# Merging with df by index_id to get the column "FTR" taht represents the full time match result and will serve as the target variable
df_prep_nf = df_prep_nf.merge(df[["index_id", "FTR"]], on="index_id", how="left")

In [ ]:
df_prep_nf.info()

Now we have a lot of features. We will firstly test all of them and then try to select only the best ones.

### Partitioning into Training & Test Data

In [ ]:
# Splitting Input features to X and target feature to y
X_nf, y_nf = df_prep_nf.set_index("index_id").drop("FTR", axis=1), df_prep_nf["FTR"]

In [ ]:
# Splitting X and y to train and test sets with a distribution of 70 / 30 and choosing random seed as 42.
X_train_nf, X_test_nf, y_train_nf, y_test_nf = train_test_split(X_nf, y_nf, test_size=0.3, random_state=42, shuffle=True)

### Scaling Data

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(X_train_nf)

X_train_nf_scaled = scaler_fitted.transform(X_train_nf)
X_test_nf_scaled = scaler_fitted.transform(X_test_nf)

## Modelling with New Features

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [5, 10, 50],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_nf = cv_procedure(estimator=estimator, param_grid=param_grid, X=X_train_nf, y=y_train_nf, scoring="accuracy")

In [ ]:
cv_results_random_forest_nf

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_nf = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_nf_scaled, y=y_train_nf)

In [ ]:
cv_results_lr_nf

## Evaluation with New Features

In [ ]:
eval_rf_nf = evaluation(cv_results_random_forest_nf, RandomForestClassifier, estimator_rf, X_train_nf, y_train_nf, X_test_nf, y_test_nf)
eval_rf_nf.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_nf, "RF_nf")

In [ ]:
eval_lr_nf = evaluation(cv_results_lr_nf, LogisticRegression, estimator_lr, X_train_nf_scaled, y_train_nf, X_test_nf_scaled, y_test_nf)
eval_lr_nf.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_nf, "LR_nf")

In [ ]:
final_results.sort_values("accuracy", ascending=False)

Unfortunately, the models with the new features (RF_nf and LR_nf) do not increase the accuracy of the existing best models RF and LR. That can be an indicator that they added too much "noise" to the data. To get rid of noise, we should select only the most important features.

# Models with selected features

- I will try to select the best features based on Univariate Correlations with the target feature.
- Univariate feature selection works by selecting the best features based on univariate statistical tests.
- It can be seen as a preprocessing step to an estimator. Scikit-learn exposes feature selection routines as objects that implement the transform method:

Source: https://scikit-learn.org/stable/modules/feature_selection.html

## Select features

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

X.shape
(150, 4)

X_new = SelectKBest(chi2, k=2).fit_transform(X, y)
X_new.shape
(150, 2)

In [ ]:
X_nf_best_3 = SelectKBest(chi2, k=3).fit_transform(X_nf, y_nf)
X_nf_best_7 = SelectKBest(chi2, k=7).fit_transform(X_nf, y_nf)
X_nf_best_20 = SelectKBest(chi2, k=20).fit_transform(X_nf, y_nf)

## Preprocessing, Modelling and Evaluation

### with k=3

In [ ]:
# Splitting X and y to train and test sets with a distribution of 70 / 30 and choosing random seed as 42.
X_train_nf_3, X_test_nf_3, y_train_nf_3, y_test_nf_3 = train_test_split(X_nf_best_3, y_nf, test_size=0.3, random_state=42, shuffle=True)

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [5, 10, 50],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_nf_3 = cv_procedure(estimator=estimator, param_grid=param_grid, X=X_train_nf_3, y=y_train_nf_3, scoring="accuracy")

In [ ]:
eval_rf_nf_3 = evaluation(cv_results_random_forest_nf_3, RandomForestClassifier, estimator_rf, X_train_nf_3, y_train_nf_3, X_test_nf_3, y_test_nf_3)
eval_rf_nf_3.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_nf_3, "RF_nf_best_3")

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(X_train_nf_3)

X_train_nf_3_scaled = scaler_fitted.transform(X_train_nf_3)
X_test_nf_3_scaled = scaler_fitted.transform(X_test_nf_3)

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_nf_3 = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_nf_3_scaled, y=y_train_nf_3)

In [ ]:
eval_lr_nf_3 = evaluation(cv_results_lr_nf_3, LogisticRegression, estimator_lr, X_train_nf_3_scaled, y_train_nf_3, X_test_nf_3_scaled, y_test_nf_3)
eval_lr_nf_3.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_nf_3, "LR_nf_best_3")

### With k=7

In [ ]:
# Splitting X and y to train and test sets with a distribution of 70 / 30 and choosing random seed as 42.
X_train_nf_7, X_test_nf_7, y_train_nf_7, y_test_nf_7 = train_test_split(X_nf_best_7, y_nf, test_size=0.3, random_state=42, shuffle=True)

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [5, 10, 50],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_nf_7 = cv_procedure(estimator=estimator, param_grid=param_grid, X=X_train_nf_7, y=y_train_nf_7, scoring="accuracy")

In [ ]:
eval_rf_nf_7 = evaluation(cv_results_random_forest_nf_7, RandomForestClassifier, estimator_rf, X_train_nf_7, y_train_nf_7, X_test_nf_7, y_test_nf_7)
eval_rf_nf_7.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_nf_7, "RF_nf_best_7")

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(X_train_nf_7)

X_train_nf_7_scaled = scaler_fitted.transform(X_train_nf_7)
X_test_nf_7_scaled = scaler_fitted.transform(X_test_nf_7)

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_nf_7 = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_nf_7_scaled, y=y_train_nf_7)

In [ ]:
eval_lr_nf_7 = evaluation(cv_results_lr_nf_7, LogisticRegression, estimator_lr, X_train_nf_7_scaled, y_train_nf_7, X_test_nf_7_scaled, y_test_nf_7)
eval_lr_nf_7.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_nf_7, "LR_nf_best_7")

### With k=20

In [ ]:
# Splitting X and y to train and test sets with a distribution of 70 / 30 and choosing random seed as 42.
X_train_nf_20, X_test_nf_20, y_train_nf_20, y_test_nf_20 = train_test_split(X_nf_best_20, y_nf, test_size=0.3, random_state=42, shuffle=True)

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [5, 10, 50],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_nf_20 = cv_procedure(estimator=estimator, param_grid=param_grid, X=X_train_nf_20, y=y_train_nf_20, scoring="accuracy")

In [ ]:
eval_rf_nf_20 = evaluation(cv_results_random_forest_nf_20, RandomForestClassifier, estimator_rf, X_train_nf_20, y_train_nf_20, X_test_nf_20, y_test_nf_20)
eval_rf_nf_20.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_nf_20, "RF_nf_best_20")

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(X_train_nf_20)

X_train_nf_20_scaled = scaler_fitted.transform(X_train_nf_20)
X_test_nf_20_scaled = scaler_fitted.transform(X_test_nf_20)

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_nf_20 = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_nf_20_scaled, y=y_train_nf_20)

In [ ]:
eval_lr_nf_20 = evaluation(cv_results_lr_nf_20, LogisticRegression, estimator_lr, X_train_nf_20_scaled, y_train_nf_20, X_test_nf_20_scaled, y_test_nf_20)
eval_lr_nf_20.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_nf_20, "LR_nf_best_20")

## Final Results

In [ ]:
final_results.sort_values("accuracy", ascending=False)

Selecting only certain features does increase the accuracy to minimal extend, but does not improve a model much better.

# New approach: Splitting into train and test based on seasons.

- Before, we created some features that are time dependent on the basis of the whole dataset. In the part of splitting the whole dataset into train and test data, we randomly sampled the dataset (shuffle).
- Although football matches can be ordered by time, the random sampling still can be done because all the information, that is time relevant, we already took into consideration in the feature creation part.
- But maybe it is still beneficial to split the data into train and test sets not randomly, but e.g. by season. I will try to do that and look what implications can be derived from that.

In [ ]:
df_prep_new_split = df_prep.merge(df[["index_id", "Season"]], on="index_id", how="left")

I will choose the year 2020 as the test set and all other season as the training sets.

In [ ]:
train_new = df_prep_new_split[df_prep_new_split["Season"] != 2020]
test_new = df_prep_new_split[df_prep_new_split["Season"] == 2020]

Xn_train = train_new.set_index("index_id").drop(["Season", "FTR"],axis=1)
Xn_test = test_new.set_index("index_id").drop(["Season", "FTR"],axis=1)

yn_train = train_new["FTR"]
yn_test = test_new["FTR"]

## Preparation, Modelling, Evaluation

In [ ]:
scaler = StandardScaler()

scaler_fitted = scaler.fit(Xn_train)

Xn_train_scaled = scaler_fitted.transform(Xn_train)
Xn_test_scaled = scaler_fitted.transform(Xn_test)

In [ ]:
# Create the parameter grid based on the restrictions from the exercise
param_grid = {
    'n_estimators': [5, 10, 50],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_new_split = cv_procedure(estimator=estimator, param_grid=param_grid, X=Xn_train, y=yn_train, scoring="accuracy")

In [ ]:
eval_rf_new_split = evaluation(cv_results_random_forest_new_split, RandomForestClassifier, estimator_rf, Xn_train, yn_train, Xn_test, yn_test)
eval_rf_new_split.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_new_split, "RF_new_split")

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_new_split = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=Xn_train_scaled, y=yn_train)

In [ ]:
eval_lr_new_split = evaluation(cv_results_lr_new_split, LogisticRegression, estimator_lr, Xn_train_scaled, yn_train, Xn_test_scaled, yn_test)
eval_lr_new_split.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_new_split, "LR_new_split")

## Final Results

In [ ]:
final_results.sort_values("accuracy", ascending=False)

With the new split strategy, the accuracy of the models (LR_new_slit & RF_new_split) does not increase, it decreases even a little bit. But it is interesting to see that the balanced accuracy score is better for these models to 1 percent point.

# Models with PCA Feature Selection

Another Feature Selection method is PCA (Principal Component Analysis). PCA is a statistical procedure that converts a set of observations of possibly correlated variables into a set of values of linearly uncorrelated variables called principal components . In simpler words, PCA is often used to simplify data, reduce noise, and find unmeasured “latent variables”. Beneath I will perform PCA to the data and train the models with the resulting pricnipal components.

## Prepare PCA

In [ ]:
# Perform PCA 
pca = PCA(.90)
pca = pca.fit(X_train_scaled)

In [ ]:
X_train_pca = pca.transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
len(X_train_pca[0])

## Perform Modelling & Evaluation with PCA

In [ ]:
# Create the parameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, 15]
    }

# Create the estimator
estimator = RandomForestClassifier(criterion="gini", random_state=42)

# Perform the experimental setup that was created in task 4.1 with the oversampled train data and with the same param_grid as in task 5.2.
cv_results_random_forest_pca = cv_procedure(estimator=estimator, param_grid=param_grid, X=X_train_pca, y=y_train)

In [ ]:
eval_rf_pca = evaluation(cv_results_random_forest_pca, RandomForestClassifier, estimator_rf, X_train_pca, y_train, X_test_pca, y_test)
eval_rf_pca.evaluate()

In [ ]:
add_eval_to_final_results(eval_rf_pca, "RF_pca")

In [ ]:
# Create the parameter grid
param_grid = {
    'C': np.logspace(-5, 8, 5),
    }

# Create the estimator
estimator_lr = LogisticRegression(random_state=42, solver='lbfgs', max_iter=400)

# Perform the experimental setup
cv_results_lr_pca = cv_procedure(estimator=estimator_lr, param_grid=param_grid, X=X_train_pca, y=y_train)

In [ ]:
eval_lr_pca = evaluation(cv_results_lr_pca, LogisticRegression, estimator_lr, X_train_pca, y_train, X_test_pca, y_test)
eval_lr_pca.evaluate()

In [ ]:
add_eval_to_final_results(eval_lr_pca, "LR_pca")

## Final Results

In [ ]:
final_results.sort_values("accuracy", ascending=False)

With PCA, the accuracy and the balanced accuracy results of the new models (RF_pca and LR_pca) decrease signficantly so in this case PCA does not improve the models.

# Conclusion

The following possibilities were tested to achieve the highest possible accuracy:
- different alogirthms (logistic regression, random forest, decision tree, k-nearest neighbors)
- hyperparameter tuning
- oversampling with two different methods (smote and simple random oversampling)
- different splitting strategies regarding train_test_split (random split and splitting by seasons)
- considering additional new features that contain information about the form of the team for multiple in-game statistics
- considering various betting odds as features (usual HDA-odds, over/under-odds, asian handicap odds)
- various feature selection methods (univariate correlation, principal component analysis, PCA (selecting best PC's))

The best model with an out-of-sample accuracy of slightly 0.53 is a random forest model where the input data includes odd-features and features that describe the short term form of a team with a feature selection method KBest. The model paramteres are {'max_depth': 5, 'n_estimators': 50}.

Although this accuracy does not appear to be high, by looking at various literature, this is not a bad accuracy:
- By using Deep Neural Networks and data of the EA Sports game Fifa, Rahman achieved an accuracy of 63,3 %. (https://link.springer.com/article/10.1007/s42452-019-1821-5) (!!!!)
- Kempa achieved an accuracy of 49,77 % by using similar form features that I used but without using odd-features. (https://towardsdatascience.com/machine-learning-algorithms-for-football-prediction-using-statistics-from-brazilian-championship-51b7d4ea0bc8)
- Razali et. al. achieved predictive accuracy of 75.09% in average across three English Premier League seasons by using Bayesian Networks.
- Various more ressources that can be found in the internt achieve a similar or even lower accuracy.

# Outlook

For the future, other algorithm that include better "memory" functions like neural networks should be tested. Also, it should be tested if its is possible to create features that can clearly indicate a draw event (D) because form my point of view, this is the weak part of all algorithms that I teste. If they could predict the draw event better, then the accuracy would automatically increase.

Also, of course, using more match data can increase model performace.